# 01 · 数据基础与探索

> **学习目标**
> 1. 搞清楚 A 股数据里最容易被忽视的三个坑：**复权、停牌、对齐**
> 2. 理解 `PricePanel` 的数据结构，以及为什么「宽表 + 字段」比长表更适合向量化研究
> 3. 学会用描述统计判断一个标的（或一个策略）的收益特征

> **运行说明**：有网络时自动拉取真实 A 股数据；无网络时自动降级为合成数据，
> 全流程仍然可以跑通。所有输出中都会标明当前数据源。

In [ ]:
import warnings

warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from qlearn.data import fetch_daily, load_panel, make_synthetic_panel
from qlearn.metrics import (
    annualized_volatility,
    cagr,
    max_drawdown,
    sharpe_ratio,
    total_return,
)

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 30)
print('环境就绪')

## 1. 先把数据拿进来

`load_panel` 会自动完成三件事：逐个标的拉取、按日期求并集对齐、组装成 `PricePanel`。
单个标的失败不会中断整体流程（真实数据源抖动是常态）。

In [ ]:
USE_REAL_DATA = True  # 改成 False 可强制离线运行
SYMBOLS = ['600519', '000858', '601318', '600036', '000001']
START, END = '2020-01-01', '2024-12-31'


def load_data():
    """优先使用真实 A 股数据，失败时降级到合成数据。"""
    if USE_REAL_DATA:
        try:
            return load_panel(SYMBOLS, start=START, end=END), '真实 A 股'
        except Exception as exc:
            print(f'[降级] 真实数据获取失败：{type(exc).__name__}: {str(exc)[:120]}')
    return make_synthetic_panel(SYMBOLS, start=START, end=END, seed=42), '合成数据（无现实意义）'


panel, data_source = load_data()
print(f'数据源  : {data_source}')
print(f'标的    : {list(panel.symbols)}')
print(f'区间    : {panel.dates[0]:%Y-%m-%d} ~ {panel.dates[-1]:%Y-%m-%d}')
print(f'交易日数: {panel.n_dates}')

## 2. 坑一：复权

**不复权价格在除权除息日会出现跳空**。这个跳空不是市场行为，却会被均线、
动量、波动率等所有指标当成真实信号，直接污染策略。

记住三条：

| 复权方式 | 适用场景 | 风险 |
|---|---|---|
| `qfq` 前复权 | 技术指标、回测（**默认选它**） | 历史价格会随最新分红而变 |
| `hfq` 后复权 | 长期收益率计算 | 价格会远高于真实成交价 |
| `raw` 不复权 | 精确复现真实成交价 | 除权日跳空污染信号 |

In [ ]:
try:
    raw = fetch_daily('600519', start='2023-01-01', end='2024-12-31', adjust='raw')
    qfq = fetch_daily('600519', start='2023-01-01', end='2024-12-31', adjust='qfq')

    ratio = (raw['close'] / qfq['close']).dropna()
    print(f'不复权 / 前复权 的比值范围 : {ratio.min():.4f} ~ {ratio.max():.4f}')
    print(f'比值累计变动               : {ratio.max() / ratio.min() - 1:.2%}')
    print()
    print('这个比值每一跳，都是一次分红送股。跳变越大，不复权价格的失真越严重。')
except Exception as exc:
    print(f'跳过（无网络）：{type(exc).__name__}: {str(exc)[:100]}')

## 3. `PricePanel` 的结构

面板把每个字段存成一张 **宽表**（行 = 交易日，列 = 标的代码），所有字段共享同一套
索引与列，从根上杜绝了对齐错误。

> 为什么不用长表（`date, symbol, close`）？
> 因为 `rolling` / `pct_change` / `shift` 这些向量化操作天然按轴工作，
> 宽表可以直接一行代码算完整个横截面，长表要先 pivot，性能与可读性都更差。

In [ ]:
print('可用字段:', list(panel.keys()))
print()
print('收盘价宽表（前 3 行）:')
panel.close.head(3)

### 3.1 NaN 的语义：不是「数据缺失」，而是「不可交易」

某标的某日没有行情，可能因为：未上市、停牌、已退市。无论哪种，**当天都不能成交**。

`panel.tradable` 给出了可交易掩码，回测引擎完全依赖它来决定当天能不能下单。

In [ ]:
print('各标的缺失收盘价的天数:')
print(panel.close.isna().sum().to_string())
print()
print('各标的可交易天数占比:')
print(panel.tradable.mean().round(4).to_string())
print()
print('注意：真实数据里若某标的缺失天数明显偏多，通常意味着长期停牌或退市，')
print('      应当用 panel.drop_inactive_symbols(min_observations=...) 剔除。')

## 4. 坑二：收益率的口径

- **简单收益** `close.pct_change()`：可加总，适合计算组合收益
- **对数收益** `log(close / close.shift(1))`：可跨期累加，适合统计建模

两者在单日幅度上非常接近，但累积起来差异会放大。本项目的回测引擎统一使用**简单收益**，
因为组合净值本来就是把各标的的简单收益按权重加权。

In [ ]:
close = panel.close
simple = close.pct_change()
log_ret = np.log(close / close.shift(1))

sample = pd.DataFrame(
    {'简单收益': simple.iloc[:, 0], '对数收益': log_ret.iloc[:, 0]}
).dropna()
print('同一日的两种收益（第一只标的）:')
display(sample.head(3))

print()
print('单日绝对涨跌幅超过 11% 的样本数（A 股涨跌停限制下，这类点往往是除权跳变或脏数据）:')
extreme = (simple.abs() > 0.11)
print(int(extreme.to_numpy().sum()))

## 5. 描述统计：先知道你在跟什么打交道

在写任何策略之前，先看清标的的基本特征：

1. **年化波动** 决定了你的仓位上限能开到多大
2. **最大回撤** 决定了你能不能拿得住（能不能扛过心理关）
3. **Sharpe** 是把收益和波动放在同一把尺子上比较

注意这里的 Sharpe 是**标的自身**的收益风险比，没有扣任何交易成本。

In [ ]:
rows = {}
for symbol in panel.symbols:
    price = close[symbol].dropna()
    rets = price.pct_change().dropna()
    rows[symbol] = {
        '累计收益': total_return(price),
        '年化收益': cagr(price),
        '年化波动': annualized_volatility(rets),
        '最大回撤': max_drawdown(price),
        'Sharpe': sharpe_ratio(rets, risk_free_rate=0.02),
    }

stats = pd.DataFrame(rows).T
print(f'各标的收益风险特征（数据源：{data_source}）:')
stats.apply(lambda col: col.map(lambda v: f'{v:.2%}') if col.name != 'Sharpe' else col.map(lambda v: f'{v:.2f}'))

In [ ]:
normalized = close.divide(close.iloc[0])

fig, ax = plt.subplots(figsize=(12, 5))
normalized.plot(ax=ax, linewidth=1.2)
ax.axhline(1.0, color='black', linewidth=0.8, linestyle=':')
ax.set_title(f'各标的净值对比（初始 = 1）｜数据源：{data_source}')
ax.set_ylabel('净值')
ax.grid(alpha=0.3, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

## 6. 坑三：相关性

「买 5 只股票」不等于「分散」——如果它们同涨同跌，你只是把 1 份风险复制成 5 份。

相关系数矩阵是判断「真分散」还是「假分散」的最快方法。

In [ ]:
corr = simple.corr()
print('日收益相关系数矩阵:')
display(corr.round(3))

values = corr.to_numpy()
n = len(values)
avg_corr = (values.sum() - np.trace(values)) / (n * n - n)
print()
print(f'平均相关系数（不含对角线）: {avg_corr:.3f}')
print('若该值 > 0.6，说明这个「组合」几乎等于单只标的，分散化收益极其有限。')

## 7. 小结与练习

**记住三句话**

1. 回测一律用 `qfq` 前复权，除非你要精确复现成交价
2. 价格 NaN = 不可交易，**绝不能**用前值填充后再拿去算信号
3. 先看清标的的波动与相关性，再决定仓位与股票池

**动手练习**

1. 把 `SYMBOLS` 换成 `qlearn.data.BUILTIN_UNIVERSES['demo_banks']`（银行板块），
   观察平均相关系数有什么变化，为什么
2. 用 `panel.drop_inactive_symbols(min_observations=250)` 剔除样本不足的标的，
   看看被剔除的是哪几只
3. 把区间缩到只有 3 个月，重新看一遍 `cagr` 与 `Sharpe`，
   体会「短样本下的年化指标有多不可信」